# PyTorch 입문 — 텐서 · 선형회귀 · 경사하강

**PyTorch** = 숫자 덩어리(텐서)를 쌓아 딥러닝 모델을 만드는 도구입니다.  
이 노트북은 [`../session01.md`](../session01.md)에서 본 감각을 **바로 실행**해 보는 입문용입니다.

> **이 노트북이 하는 일**  
> 텐서를 만들고, 미분을 한 번 돌려 본 뒤, `y = wx + b`를 손으로·모듈로 각각 학습합니다.

환경: `torch-env` 커널을 쓰는 것을 권장합니다. (`session01` 9절 참고)


## 1. 준비 — import · 버전 · device

**device** = 계산을 어디서 할지입니다. 이 입문에서는 **CPU**만 씁니다.


In [1]:
import torch

# 설치·환경이 맞는지 버전으로 먼저 확인합니다
print("torch 버전:", torch.__version__)

# GPU가 있어도 입문 실습은 CPU로 통일합니다
device = torch.device("cpu")
print("device:", device)


torch 버전: 2.5.1
device: cpu


## 2. 텐서 만들기 · shape · NumPy 변환

**텐서(Tensor)** = NumPy 배열과 비슷한 **숫자 상자**입니다.  
**shape** = 상자가 몇 줄·몇 칸인지입니다.


In [2]:
# 1차원 텐서 = 숫자 한 줄
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("x:", x)
print("shape:", x.shape)
print("dtype:", x.dtype)

# 2줄 표(2차원)도 같은 방식으로 만듭니다
mat = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("mat shape:", mat.shape)

# NumPy로 갔다가 다시 텐서로 — 서로 왕복할 수 있습니다
x_np = x.numpy()
x_back = torch.from_numpy(x_np)
print("NumPy:", x_np)
print("다시 텐서:", x_back)


x: tensor([1., 2., 3., 4.])
shape: torch.Size([4])
dtype: torch.float32
mat shape: torch.Size([2, 2])
NumPy: [1. 2. 3. 4.]
다시 텐서: tensor([1., 2., 3., 4.])


## 3. 연산 · requires_grad · backward

**requires_grad=True** = "이 값으로 미분할 거니 계산 과정을 기억해 주세요"라는 표시입니다.  
**backward** = 기억해 둔 계산을 거꾸로 따라가며 **기울기(gradient)** 를 구하는 호출입니다.


In [3]:
# w를 학습할 후보로 두고, 미분 추적을 켭니다
w = torch.tensor(2.0, requires_grad=True)

# 간단한 식: loss = (w * 3 - 4)^2  → w=2일 때 값을 계산
y = w * 3.0
loss = (y - 4.0) ** 2
print("loss:", loss.item())

# 한 번만 backward — w.grad에 d(loss)/d(w)가 쌓입니다
loss.backward()
print("w.grad:", w.grad)


loss: 4.0
w.grad: tensor(12.)


## 4. 수동 경사하강 — `y = wx + b`

**경사하강** = 오차(손실)가 작아지도록 `w`, `b`를 조금씩 고치는 반복입니다.  
**MSE** = 예측과 정답 차이의 제곱 평균(평균 제곱 오차)입니다.

정답은 `y = 2x + 1`로 두고, 모델이 `w`, `b`를 스스로 맞추게 합니다.


In [4]:
torch.manual_seed(0)

# 입력 x와 정답 y (진짜 식: y = 2x + 1)
x = torch.linspace(-1.0, 1.0, 20).unsqueeze(1)  # 열 벡터로
y_true = 2.0 * x + 1.0

# 시작할 때 w, b는 아무 값이나 — 학습으로 맞춥니다
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
lr = 0.1  # 학습률 = 한 번에 얼마나 고칠지

for step in range(80):
    y_pred = w * x + b
    # MSE = 차이의 제곱을 평균
    loss = ((y_pred - y_true) ** 2).mean()

    # 이전 기울기를 지운 뒤 새로 계산합니다
    if w.grad is not None:
        w.grad.zero_()
    if b.grad is not None:
        b.grad.zero_()

    loss.backward()

    # 추적 밖에서 값만 갱신 (이 줄 자체는 미분 대상이 아님)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    if step % 20 == 0 or step == 79:
        print(f"step {step:3d} | loss={loss.item():.6f} | w={w.item():.3f} | b={b.item():.3f}")

print("목표: w≈2, b≈1")


step   0 | loss=1.750579 | w=1.575 | b=-0.035
step  20 | loss=0.003856 | w=1.908 | b=0.988
step  40 | loss=0.000170 | w=1.980 | b=1.000
step  60 | loss=0.000008 | w=1.996 | b=1.000
step  79 | loss=0.000000 | w=1.999 | b=1.000
목표: w≈2, b≈1


## 5. (선택) `nn.Linear` + `optim.SGD`

같은 문제를 **모듈**과 **옵티마이저**로 풀면, 손으로 `w`/`b`를 갱신하지 않아도 됩니다.  
`nn.Linear(1, 1)` = 입력 1개 → 출력 1개인 직선 `y = wx + b`입니다.


In [5]:
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(1)

x = torch.linspace(-1.0, 1.0, 20).unsqueeze(1)
y_true = 2.0 * x + 1.0

model = nn.Linear(1, 1)  # 내부적으로 w, b를 가짐
criterion = nn.MSELoss()  # 손실 = MSE
optimizer = optim.SGD(model.parameters(), lr=0.1)

for step in range(80):
    y_pred = model(x)
    loss = criterion(y_pred, y_true)

    optimizer.zero_grad()  # 이전 기울기 비우기
    loss.backward()
    optimizer.step()  # w, b 갱신

    if step % 20 == 0 or step == 79:
        w = model.weight.item()
        b = model.bias.item()
        print(f"step {step:3d} | loss={loss.item():.6f} | w={w:.3f} | b={b:.3f}")

print("목표: w≈2, b≈1")


step   0 | loss=2.889735 | w=0.625 | b=-0.153
step  20 | loss=0.038296 | w=1.702 | b=0.987
step  40 | loss=0.001780 | w=1.936 | b=1.000
step  60 | loss=0.000083 | w=1.986 | b=1.000
step  79 | loss=0.000005 | w=1.997 | b=1.000
목표: w≈2, b≈1


## 정리

1. 텐서는 NumPy처럼 생긴 숫자 상자입니다.
2. `requires_grad` + `backward`로 기울기를 구합니다.
3. 경사하강은 "손실을 줄이는 방향"으로 `w`, `b`를 고칩니다.
4. `nn.Linear` + `optim.SGD`는 같은 일을 모듈이 대신합니다.

이론 복습: [`../session01.md`](../session01.md)
